# Confidence Calibration — DeBERTa Soft-Label Model

**Prerequisite:** Run `error_analysis.ipynb` cells 1–29 (through the `14-LOAD` cell and `A-0` setup)
so that `dev_df`, `dev_probs`, `dev_labels`, `final_preds`, `best_thresh`, and `DEVICE` are all in scope.

Alternatively, run the `14-LOAD` cell below first, then the `A-0` setup cell, then Analysis 4.

In [ ]:
# ── 14-LOAD: Restore all variables after a kernel restart ──────────────────
import os, random, logging
from pathlib import Path
from collections import Counter
from urllib import request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score

logging.basicConfig(level=logging.WARNING)
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED      = 777
MAX_LEN   = 256
DROPOUT   = 0.1
THRESHOLD = 0.35
CKPT_PATH = 'best_model_pcl.pt'   # adjust path as needed
DATA_DIR  = './dpm_data'          # adjust path as needed

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

class PCLClassifier(nn.Module):
    def __init__(self, model_name, hidden_size=256, dropout=DROPOUT):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        enc_hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(enc_hidden, hidden_size), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_size, 1),
        )
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           token_type_ids=token_type_ids)
        return self.classifier(out.last_hidden_state[:, 0, :]).squeeze(-1)

class PCLDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN, use_soft_labels=False):
        self.texts         = df['text'].tolist()
        self.labels        = df['label'].astype(float).tolist()
        self.binary_labels = df['label'].tolist()
        self.soft_targets  = df.get('soft_target', df['label'].astype(float)).tolist()
        self.tokenizer     = tokenizer
        self.max_len       = max_len
        self.use_soft      = use_soft_labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], padding='max_length', truncation=True,
                             max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'soft_label':     torch.tensor(self.soft_targets[idx] if self.use_soft else self.labels[idx], dtype=torch.float),
            'binary_label':   torch.tensor(self.binary_labels[idx], dtype=torch.long),
        }

# Load checkpoint
print(f'Loading checkpoint: {CKPT_PATH}')
ckpt       = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
hp         = ckpt['hyperparameters']
tokenizer  = AutoTokenizer.from_pretrained(hp['model_name'])
model      = PCLClassifier(model_name=hp['model_name'], dropout=DROPOUT).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
best_thresh = float(ckpt.get('optimal_threshold', THRESHOLD))
print(f'Loaded  |  threshold={best_thresh:.2f}  |  device={DEVICE}')

# Reload dev DataFrame
SOFT_TARGET_MAP = {0: 0.00, 1: 0.10, 2: 0.70, 3: 0.90, 4: 1.00}
if not Path('dont_patronize_me.py').exists():
    url = 'https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py'
    with request.urlopen(url) as f, open('dont_patronize_me.py', 'w') as out:
        out.write(f.read().decode('utf-8'))
from dont_patronize_me import DontPatronizeMe

dpm = DontPatronizeMe(DATA_DIR, DATA_DIR)
dpm.load_task1()
data = dpm.train_task1_df
if 'orig_label' not in data.columns:
    raw = pd.read_csv(os.path.join(DATA_DIR, 'dontpatronizeme_pcl.tsv'), sep='\t', skiprows=4,
                      names=['row_id','par_id','keyword','country','text','orig_label'],
                      dtype={'par_id': str, 'orig_label': 'Int64'}, keep_default_na=False)
    raw['orig_label'] = raw['orig_label'].astype(int)
    data = data.merge(raw[['par_id','orig_label']], on='par_id', how='left')

teids = pd.read_csv(os.path.join(DATA_DIR, 'dev_semeval_parids-labels.csv'))
teids.par_id = teids.par_id.astype(str)

def build_df(ids_df, source_df):
    rows = []
    for parid in ids_df.par_id:
        row = source_df.loc[source_df.par_id == parid]
        if len(row) == 0: continue
        rows.append({'par_id': parid, 'keyword': row.keyword.values[0],
                     'text': row.text.values[0], 'label': int(row.label.values[0]),
                     'orig_label': int(row.orig_label.values[0])})
    return pd.DataFrame(rows)

tedf = build_df(teids, data)
tedf['soft_target'] = tedf['orig_label'].map(SOFT_TARGET_MAP)

# Run inference
dev_dataset = PCLDataset(tedf, tokenizer, max_len=MAX_LEN, use_soft_labels=False)
dev_loader  = DataLoader(dev_dataset, batch_size=32, shuffle=False, num_workers=0)
all_logits, all_labels = [], []
with torch.no_grad():
    for batch in dev_loader:
        logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        all_logits.append(logits.cpu().numpy())
        all_labels.append(batch['binary_label'].numpy())
dev_logits  = np.concatenate(all_logits)
dev_labels  = np.concatenate(all_labels)
dev_probs   = torch.sigmoid(torch.tensor(dev_logits)).numpy()
final_preds = (dev_probs >= best_thresh).astype(int)

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader_analyzer = SentimentIntensityAnalyzer()
tedf_copy = tedf.copy()
tedf_copy['vader_compound'] = tedf_copy['text'].apply(
    lambda t: vader_analyzer.polarity_scores(str(t))['compound'])
tedf_copy['true_label'] = dev_labels.astype(int)
tedf_copy['pred_label'] = final_preds.astype(int)
tedf_copy['pred_prob']  = dev_probs
dev_df = tedf_copy.copy().rename(columns={'pred_prob': 'prob'})

f1_macro = f1_score(dev_labels, final_preds, average='macro', zero_division=0)
print(f'Dev Macro-F1: {f1_macro:.4f}  |  Threshold: {best_thresh:.2f}')
print('All variables ready for calibration analysis.')

## Analysis 4: Confidence Calibration Plot

Bins all dev examples into 10 equal-width probability buckets and compares
the fraction of actual PCL examples in each bucket to the predicted probability.
A well-calibrated model → points close to y=x.
Also computes Expected Calibration Error (ECE).

In [ ]:
import matplotlib.pyplot as plt
import os

TEAL   = '#008080'
SALMON = '#FA8072'
os.makedirs('./figures', exist_ok=True)

n_bins    = 10
bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
bin_lower = bin_edges[:-1]
bin_upper = bin_edges[1:]
bin_mid   = (bin_lower + bin_upper) / 2.0

actual_fracs, pred_fracs, bin_sizes = [], [], []
for lo, hi in zip(bin_lower, bin_upper):
    mask = (dev_df['prob'] >= lo) & (dev_df['prob'] < hi)
    if mask.sum() == 0:
        actual_fracs.append(np.nan); pred_fracs.append(lo + 0.05); bin_sizes.append(0)
    else:
        actual_fracs.append(dev_df.loc[mask, 'true_label'].mean())
        pred_fracs.append(dev_df.loc[mask, 'prob'].mean())
        bin_sizes.append(mask.sum())

actual_fracs = np.array(actual_fracs)
pred_fracs   = np.array(pred_fracs)
bin_sizes    = np.array(bin_sizes)

valid = ~np.isnan(actual_fracs)
ece   = np.sum(bin_sizes[valid] * np.abs(actual_fracs[valid] - pred_fracs[valid])) / len(dev_df)
print(f'Expected Calibration Error (ECE): {ece:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.5, label='Perfect calibration')
sizes = np.where(valid, np.clip(bin_sizes / max(bin_sizes), 0.1, 1) * 300, 0)
ax.scatter(pred_fracs[valid], actual_fracs[valid], s=sizes[valid], color=TEAL,
           edgecolors='#333333', linewidths=0.8, zorder=3, label='Calibration bins')
for pf, af in zip(pred_fracs[valid], actual_fracs[valid]):
    ax.plot([pf, pf], [pf, af], lw=1.5, color=SALMON, alpha=0.6)
ax.axvline(best_thresh, color='#888888', linestyle=':', lw=1.5,
           label=f'Decision threshold = {best_thresh}')
ax.set_xlabel('Mean predicted probability (bin)', fontsize=10)
ax.set_ylabel('Actual PCL fraction (bin)', fontsize=10)
ax.set_title(f'Reliability Diagram\nECE = {ece:.4f}  (lower is better)', fontsize=10)
ax.legend(fontsize=9)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.3)
ax.text(0.05, 0.92, f'ECE = {ece:.4f}', transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax2 = axes[1]
ax2.hist(dev_df[dev_df.true_label == 0]['prob'], bins=30, color=SALMON, alpha=0.7,
         label='True  No-PCL', density=True)
ax2.hist(dev_df[dev_df.true_label == 1]['prob'], bins=30, color=TEAL, alpha=0.7,
         label='True  PCL', density=True)
ax2.axvline(best_thresh, color='#333333', linestyle='--', lw=2,
            label=f'Threshold = {best_thresh}')
ax2.set_xlabel('Predicted probability'); ax2.set_ylabel('Density')
ax2.set_title('Probability Separation by True Class')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
fig.savefig('./figures/calibration_plot.png', dpi=150)
plt.show()
print('Saved → ./figures/calibration_plot.png')

cal_df = pd.DataFrame({
    'bin_lower': bin_lower.round(2), 'bin_upper': bin_upper.round(2),
    'n_examples': bin_sizes,
    'mean_predicted_prob': np.round(pred_fracs, 4),
    'actual_pcl_fraction': np.round(actual_fracs, 4),
    'gap': np.round(np.abs(actual_fracs - pred_fracs), 4),
})
print(cal_df.to_string(index=False))